# BI Intern Assignment — D2C Analytics
**Approach:** load the 4 sheets, drop the trailing blank/note rows in `orders`, `customers`, `product_catalog`, `product_pricing` (PK/FK footnotes), then run SQL on an in-memory SQLite DB.

**Revenue definition:** `quantity × MRP × (1 − discount_pct/100)`, where discount comes from `product_pricing` joined on **both** `product_name` and `channel`.

In [1]:
import pandas as pd, sqlite3
f = 'bi_intern_assignment.xlsx'
orders    = pd.read_excel(f, sheet_name='📊 orders').dropna(subset=['customer_id'])          # drops blank + footnote rows
customers = pd.read_excel(f, sheet_name='👤 customers').dropna(subset=['customer_name'])
catalog   = pd.read_excel(f, sheet_name='🗂 product_catalog').dropna(subset=['brand'])
pricing   = pd.read_excel(f, sheet_name='💰 product_pricing').dropna(subset=['channel'])
orders['order_id'] = orders.order_id.astype(int); orders['quantity'] = orders.quantity.astype(int)

con = sqlite3.connect(':memory:')
for n, d in [('orders',orders),('customers',customers),('product_catalog',catalog),('product_pricing',pricing)]:
    d.to_sql(n, con, index=False)
q = lambda sql: pd.read_sql(sql, con)

# sanity checks: keys line up, nothing lost in joins
print(len(orders), len(customers), len(catalog), len(pricing))
print('orders w/o customer :', q('select count(*) n from orders o left join customers c using(customer_id) where c.customer_id is null').n[0])
print('orders w/o discount :', q('select count(*) n from orders o left join product_pricing p using(product_name,channel) where p.discount_pct is null').n[0])

50 26 18 90
orders w/o customer : 0
orders w/o discount : 0


## SQL
### QS1 — orders & avg MRP per customer segment

In [2]:
q('''
SELECT c.customer_segment,
       COUNT(*)            AS total_orders,
       ROUND(AVG(o.MRP),2) AS avg_MRP
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.customer_segment
ORDER BY total_orders DESC
''')

,customer_segment,total_orders,avg_MRP
0,Gold,22,1327.64
1,Silver,18,1282.11
2,Bronze,10,1282.90


### QS2 — total revenue per product (after channel discount)

In [3]:
q('''
SELECT o.product_name,
       ROUND(SUM(o.quantity * o.MRP * (1 - p.discount_pct/100.0)), 2) AS total_revenue
FROM orders o
JOIN product_pricing p ON p.product_name = o.product_name AND p.channel = o.channel   -- join on BOTH keys
GROUP BY o.product_name
ORDER BY total_revenue DESC
''')

,product_name,total_revenue
0,Vitamin C Serum,30730.35
1,Collagen Supplement,20429.18
2,Conditioner,14298.75
3,Ashwagandha,14100.00
4,Night Cream,13540.80
5,Moisturizer,13172.60
6,Hair Mask,12145.20
7,Vitamin D3,10751.58
8,Eyeliner,10709.26
9,Probiotic Blend,10605.76


### QS3 — channel with the highest discount_pct (per orders made)
"Highest discount" is ambiguous, so I show both readings: **average** discount across orders placed, and the **max** discount applied on any order. Both give the same channel.

In [4]:
q('''
SELECT o.channel,
       COUNT(*)                     AS orders,
       ROUND(AVG(p.discount_pct),2) AS avg_discount_pct,
       MAX(p.discount_pct)          AS max_discount_pct
FROM orders o
JOIN product_pricing p ON p.product_name = o.product_name AND p.channel = o.channel
GROUP BY o.channel
ORDER BY avg_discount_pct DESC
''')

,channel,orders,avg_discount_pct,max_discount_pct
0,Amazon,7,17.43,22.0
1,Instagram,11,11.36,15.0
2,Website,8,9.25,12.0
3,WhatsApp,9,7.11,10.0
4,App,15,5.53,10.0


## Python
### QP1 — Category | Total Revenue | Orders | Avg Order Value

In [5]:
df = orders.merge(pricing, on=['product_name','channel'], how='left', validate='m:1')
df['revenue'] = df.quantity * df.MRP * (1 - df.discount_pct/100)

cat_summary = (df.groupby('category')
                 .agg(**{'Total Revenue':('revenue','sum'), 'Orders':('order_id','nunique')})
                 .assign(**{'Avg Order Value': lambda x: x['Total Revenue']/x['Orders']})
                 .round(2).sort_values('Total Revenue', ascending=False)
                 .reset_index().rename(columns={'category':'Category'}))
cat_summary

,Category,Total Revenue,Orders,Avg Order Value
0,Wellness,65337.06,16,4083.57
1,Skincare,64722.59,15,4314.84
2,Haircare,38013.82,14,2715.27
3,Makeup,15283.39,5,3056.68


## Task 3 — LLM dashboard
### Exact initial prompt given to the LLM
> Build me a single-page HTML dashboard for a D2C beauty & wellness brand's channel and product performance. Requirements: one self-contained .html file, vanilla JS + inline CSS/SVG only (no CDN, no server, works offline, opens by double-click). Data (embed as a JSON array in the file): 50 orders with order_date, category, product_name, channel (App/Website/Instagram/Amazon/WhatsApp), quantity, MRP, discount_pct (per product+channel), cost_price. Net revenue = quantity x MRP x (1 - discount_pct/100). Include: 4 KPI cards (net revenue, orders, avg order value, gross margin); revenue by channel; avg discount vs gross margin by channel; monthly revenue trend; revenue by category; a sortable product table (orders, units, net revenue, avg discount, margin); category and channel dropdown filters that update everything. Clean, minimal, mobile-responsive, supports dark mode. Keep it to something a manager can read in 30 seconds.

### Dashboard
Self-contained single HTML file (no server, no external libraries): `dashboard.html` — LINK_PLACEHOLDER